***
# SECTION 0: DATA DESIGN
***

1. Firstly, read the 12 months of New York taxi csv files
    - sourced from: <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>
2. Fetch a New York weather dataset
    - documentation: <https://mesonet.agron.iastate.edu/nws/cf6table.php?station=KNYC&opt=bystation&year=2024>
    - API call: <https://mesonet.agron.iastate.edu/json/cf6.py?station=KNYC&year=2024>
    - originally reported by the *US National Weather Service*
3. Add a hard-coded calendar
    - sourced from: <https://www.officeholidays.com/countries/usa/new-york/2024>
4. Preprocess the data as training, validation & test sets

## **[0.1]** Define dataset

Includes:
1. Reading csv files
2. HTTP-fetching weather data
3. Hard-coded calendar data
4. Final aggregation, by `route`, `time_of_day` and `date`

In [1]:
seed = 123  # random state seed

import pandas as pd
import requests
import numpy as np
from functools import lru_cache
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder



# Set this to the folder that contains your CSV files.
# Examples:
#   DATA_DIR = Path("./data")
#   DATA_DIR = Path("../data")
#   DATA_DIR = Path("/full/path/to/data")
DATA_DIR = Path("./data")


def resolve_data_path(filename):
    """Resolve a file inside the project data directory with a clear error message."""
    candidate = (DATA_DIR / filename).resolve()
    if candidate.exists():
        return candidate

    # Common notebook-location fallback
    fallback = (Path.cwd() / "data" / filename).resolve()
    if fallback.exists():
        return fallback

    searched = [str(candidate), str(fallback)]
    raise FileNotFoundError(
        f"Could not find {filename}. Looked in: {searched}. "
        "Update DATA_DIR near the top of the notebook to point at your data folder."
    )

##############################################################################
# CLEANEST DATA PIPELINE DESIGN
#
# 1. read_single_month(month)        -> read one raw CSV
# 2. clean_trip_data(df, zones)      -> clean / transform one dataframe
# 3. read_all_trips(start, end)      -> read + clean all months ONCE
# 4. add_external_features(df)       -> merge weather + calendar ONCE
# 5. build_shared_base(start, end)   -> one shared enriched trip table
# 6. read_agg / read_fare_data /
#    read_revenue_data               -> derive modelling tables from shared base
#
# This keeps the structure explicit:
# - raw files are read once
# - one shared cleaned base is built once
# - demand / fare / revenue datasets are derived from that base
##############################################################################

def get_time_of_day(hour):
    if 0 <= hour < 6:
        return "Late Night"
    elif 6 <= hour < 10:
        return "Morning Rush"
    elif 10 <= hour < 16:
        return "Midday"
    elif 16 <= hour < 20:
        return "Evening Rush"
    else:
        return "Evening"


def read_single_month(month=1):
    """Read one raw monthly taxi CSV from disk."""
    path = resolve_data_path(f"nyc_taxi_2024-{month:02d}.csv")
    return pd.read_csv(
        path,
        parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"],
        dtype={"store_and_fwd_flag": str},
    )


@lru_cache(maxsize=1)
def read_zone_lookup():
    """Read zone lookup once per kernel session."""
    return pd.read_csv(resolve_data_path("taxi_zone_lookup.csv"))


def clean_trip_data(df, zones):
    """Clean and transform a raw trip dataframe that is already in memory."""
    df = df.copy()

    # ------------------------------ CLEANING -------------------------------- #
    df = df[df["fare_amount"] > 0]
    df = df[df["trip_distance"] > 0]
    df = df[df["tpep_pickup_datetime"] < df["tpep_dropoff_datetime"]]
    df = df.dropna(subset=["tpep_pickup_datetime"])

    # Keep only Manhattan pickups for this project
    manhattan_ids = zones.loc[zones["Borough"] == "Manhattan", "LocationID"]
    df = df[df["PULocationID"].isin(manhattan_ids)]

    # --------------------------- TIME FEATURES ------------------------------ #
    df["pickup_date"] = pd.to_datetime(df["tpep_pickup_datetime"].dt.date)
    df["pickup_hr"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_day"] = df["tpep_pickup_datetime"].dt.day
    df["pickup_dow"] = df["tpep_pickup_datetime"].dt.weekday
    df["pickup_week"] = df["tpep_pickup_datetime"].dt.isocalendar().week.astype(int)
    df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
    df["pickup_year"] = df["tpep_pickup_datetime"].dt.year
    df["time_of_day"] = df["pickup_hr"].apply(get_time_of_day)

    # -------------------------- SIMPLE MAPPINGS ----------------------------- #
    df["store_and_fwd_flag"] = df["store_and_fwd_flag"].map({"Y": True, "N": False})

    # --------------------------- ZONE ENRICHMENT ---------------------------- #
    pickup_lookup = zones[["LocationID", "Zone"]].rename(columns={"Zone": "pickup_zone"})
    dropoff_lookup = zones[["LocationID", "Zone"]].rename(columns={"Zone": "dropoff_zone"})

    df = df.merge(pickup_lookup, left_on="PULocationID", right_on="LocationID", how="left")
    df = df.drop(columns=["LocationID"])

    df = df.merge(dropoff_lookup, left_on="DOLocationID", right_on="LocationID", how="left")
    df = df.drop(columns=["LocationID"])

    df["route"] = df["pickup_zone"].astype(str) + " to " + df["dropoff_zone"].astype(str)

    # Trip duration is useful for fare / revenue modelling
    df["trip_duration_mins"] = (
        (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    ).clip(lower=1)

    # ---------------------------- FINAL TIDY-UP ----------------------------- #
    df = df.drop(columns=[
        "RatecodeID",
        "VendorID",
        "PULocationID",
        "DOLocationID",
        "store_and_fwd_flag",
    ])

    df = df.astype({
        "pickup_zone": "category",
        "dropoff_zone": "category",
        "route": "category",
    })

    df = df[df["pickup_year"] == 2024]
    return df


@lru_cache(maxsize=None)
def read_all_trips(month_start=1, month_end=12):
    """Read and clean the raw monthly taxi files exactly once per argument pair."""
    zones = read_zone_lookup()
    monthly_frames = []

    for month in range(month_start, month_end + 1):
        print(f"reading trip data for month {month}")
        raw_df = read_single_month(month)
        clean_df = clean_trip_data(raw_df, zones)
        monthly_frames.append(clean_df)

    return pd.concat(monthly_frames, ignore_index=True)


@lru_cache(maxsize=1)
def read_weather_data():
    """Read and clean weather data once per kernel session."""
    response = requests.get("https://mesonet.agron.iastate.edu/json/cf6.py?station=KNYC&year=2024")
    response.raise_for_status()
    response_json = response.json()

    wthr = pd.DataFrame(response_json["results"])
    wthr["date"] = pd.to_datetime(wthr["valid"])
    wthr = wthr.drop(columns=[
        "name", "station", "valid", "state", "wfo", "link", "product",
        "minutes_sunshine", "possible_sunshine", "hdd", "cdd",
        "gust_drct", "avg_drct", "snowd_12z", "avg_smph"
    ])

    wthr["fog"] = wthr["wxcodes"].str.contains("1", na=False).astype(bool)
    wthr["low_vis"] = wthr["wxcodes"].str.contains("2", na=False).astype(bool)
    wthr["thunder"] = wthr["wxcodes"].str.contains("3", na=False).astype(bool)
    wthr["ice"] = wthr["wxcodes"].str.contains("4", na=False).astype(bool)
    wthr["hail"] = wthr["wxcodes"].str.contains("5", na=False).astype(bool)
    wthr["freezing_rain"] = wthr["wxcodes"].str.contains("6", na=False).astype(bool)
    wthr["duststorm"] = wthr["wxcodes"].str.contains("7", na=False).astype(bool)
    wthr["haze"] = wthr["wxcodes"].str.contains("8", na=False).astype(bool)
    wthr["blowing_snow"] = wthr["wxcodes"].str.contains("9", na=False).astype(bool)
    wthr["tornado"] = wthr["wxcodes"].str.contains("X", na=False).astype(bool)

    wthr.replace("T", 0, inplace=True)
    wthr.replace("M", np.nan, inplace=True)
    wthr = wthr.infer_objects(copy=False)
    wthr = wthr.astype({
        "snow": "float64",
        "precip": "float64",
    })

    wthr = wthr.drop(columns=[
        "wxcodes",
        "ice",
        "tornado",
        "blowing_snow",
        "duststorm",
        "avg_temp",
        "dep_temp",
        "low",
        "gust_smph",
    ])

    wthr = wthr.rename(columns={
        "high": "temp_high",
        "max_smph": "max_wind_speed",
        "cloud_ss": "cloud_coverage",
    })

    wthr["max_wind_speed"] = wthr["max_wind_speed"].interpolate(method="linear")
    return wthr


@lru_cache(maxsize=1)
def read_calendar_data():
    """Read calendar / holiday data once per kernel session."""
    holidays = {
        "date": [
            "2024-01-01", "2024-01-15", "2024-02-12", "2024-02-19",
            "2024-03-31", "2024-05-12", "2024-05-27", "2024-06-16",
            "2024-06-19", "2024-07-04", "2024-09-02", "2024-10-14",
            "2024-10-31", "2024-11-05", "2024-11-11", "2024-11-28",
            "2024-12-25", "2024-12-31"
        ],
        "holiday": [
            "New Year's Day", "Martin Luther King Jr. Day", "Lincoln's Birthday",
            "Washington's Birthday", "Easter Sunday", "Mother's Day",
            "Memorial Day", "Father's Day", "Juneteenth", "Independence Day",
            "Labor Day", "Columbus Day", "Halloween", "Election Day",
            "Veterans Day", "Thanksgiving", "Christmas Day", "New Year's Eve"
        ],
        "holiday_type": [
            "National", "National", "Government", "Regional", "Not a holiday",
            "Not a holiday", "National", "Not a holiday", "National", "National",
            "National", "Regional", "Observance", "Observance", "National",
            "National", "National", "Observance"
        ],
    }
    cal = pd.DataFrame(holidays)
    cal["date"] = pd.to_datetime(cal["date"])
    return cal


def add_external_features(df):
    """Merge weather and holiday signals onto an existing trip/aggregate dataframe."""
    weather = read_weather_data()
    cal = read_calendar_data()

    out = df.merge(weather, left_on="pickup_date", right_on="date", how="left")
    out = out.drop(columns=["date"])

    out = out.merge(cal, left_on="pickup_date", right_on="date", how="left")
    out["holiday"] = out["holiday"].fillna("None")
    out = out.drop(columns=["date", "holiday_type"])

    return out


@lru_cache(maxsize=None)
def build_shared_base(month_start=1, month_end=12):
    """Build one shared enriched trip table for all downstream models."""
    trips = read_all_trips(month_start, month_end)
    base = add_external_features(trips)
    return base


groupings = [
    "pickup_month",
    "pickup_week",
    "pickup_date",
    "pickup_dow",
    "time_of_day",
    "pickup_zone",
    "dropoff_zone",
    "route",
]


def add_lag_demand(df):
    """Add previous-period demand by route and time of day."""
    daily = df.groupby(["pickup_date", "time_of_day", "route"], as_index=False)["total_ride_count"].sum()
    daily = daily.sort_values(["route", "time_of_day", "pickup_date"])
    daily["lag_demand"] = daily.groupby(["route", "time_of_day"])["total_ride_count"].shift(1)
    out = df.merge(
        daily[["route", "pickup_date", "time_of_day", "lag_demand"]],
        on=["route", "pickup_date", "time_of_day"],
        how="left",
    )
    out["lag_demand"] = out["lag_demand"].fillna(0)
    return out


def read_agg(month_start=1, month_end=12, groupings=groupings):
    """Demand modelling table derived from the shared base."""
    base = build_shared_base(month_start, month_end)

    agg_df = base.groupby(groupings, as_index=False, observed=True).agg(
        total_ride_count=("tpep_pickup_datetime", "count"),
        avg_fare_amount=("fare_amount", "mean"),
    )

    agg_df = add_external_features(agg_df)
    agg_df = add_lag_demand(agg_df)
    return agg_df

fare_groupings = [
    "pickup_date",
    "time_of_day",
    "pickup_zone",
]

revenue_groupings = [
    "pickup_date",
    "time_of_day",
    "pickup_zone",
]


def read_fare_data(month_start=1, month_end=12, groupings=fare_groupings):
    base = build_shared_base(month_start, month_end)

    fare_df = (
        base.groupby(groupings, as_index=False, observed=True)
        .agg(
            trip_count=("tpep_pickup_datetime", "count"),
            avg_fare_amount=("fare_amount", "mean"),
            median_fare_amount=("fare_amount", "median"),
            avg_trip_distance=("trip_distance", "mean"),
            avg_duration_mins=("trip_duration_mins", "mean"),
            avg_tip_amount=("tip_amount", "mean"),
        )
    )

    fare_df = add_external_features(fare_df)
    fare_df = add_lag_demand(fare_df)
    return fare_df


def read_revenue_data(month_start=1, month_end=12, groupings=revenue_groupings):
    base = build_shared_base(month_start, month_end)

    revenue_df = (
        base.groupby(groupings, as_index=False, observed=True)
        .agg(
            trip_count=("tpep_pickup_datetime", "count"),
            total_revenue=("total_amount", "sum"),
            total_fare_amount=("fare_amount", "sum"),
            total_tip_amount=("tip_amount", "sum"),
            avg_fare_amount=("fare_amount", "mean"),
            avg_trip_distance=("trip_distance", "mean"),
            avg_duration_mins=("trip_duration_mins", "mean"),
        )
    )

    revenue_df = add_external_features(revenue_df)
    revenue_df = add_lag_demand(revenue_df)
    return revenue_df



In [2]:
# Read the data in
df = read_agg(month_start=1, month_end=12)
df.head(3)

reading trip data for month 1
reading trip data for month 2
reading trip data for month 3
reading trip data for month 4
reading trip data for month 5
reading trip data for month 6
reading trip data for month 7
reading trip data for month 8
reading trip data for month 9
reading trip data for month 10
reading trip data for month 11
reading trip data for month 12


C:\Users\Emma Davidson\AppData\Local\Temp\ipykernel_1144\3800587077.py:191: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  wthr = wthr.infer_objects(copy=False)


,pickup_month,pickup_week,pickup_date,pickup_dow,time_of_day,pickup_zone,dropoff_zone,route,total_ride_count,avg_fare_amount,...,max_wind_speed,cloud_coverage,fog,low_vis,thunder,hail,freezing_rain,haze,holiday,lag_demand
0,1,1,2024-01-01,0,Evening,Alphabet City,Gramercy,Alphabet City to Gramercy,1,7.90,...,9.0,7.0,False,False,False,False,False,False,New Year's Day,0.0
1,1,1,2024-01-01,0,Evening,Alphabet City,Meatpacking/West Village West,Alphabet City to Meatpacking/West Village West,1,13.50,...,9.0,7.0,False,False,False,False,False,False,New Year's Day,0.0
2,1,1,2024-01-01,0,Evening,Alphabet City,Penn Station/Madison Sq West,Alphabet City to Penn Station/Madison Sq West,1,19.78,...,9.0,7.0,False,False,False,False,False,False,New Year's Day,0.0


## **[0.2]** Preprocessing

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Recast object types -> category (just to be sure)
df = df.astype({
    'pickup_zone': 'category',
    'route': 'category',
    'time_of_day': 'category',
    'hail': 'bool',  
    'freezing_rain': 'bool',  
    'holiday': 'category',
})

# Remove waste columns
df = df.drop(errors='ignore', columns=[
    # 'lag_demand',
    'dropoff_zone',
    'pickup_date',
    'service_route',
    'pickup_service_zone',
    'dropoff_service_zone',
    'pickup_week',
    'pickup_day',  
    'avg_fare_amount',        
    # 'hail',
    # 'freezing_rain',
    # 'haze',
])

# Transformations
df['precip'] = df['total_ride_count'] * df['precip'] # Scale precipitaion with lag demand
df['temp_high'] = df['total_ride_count'] * df['temp_high'] # Scale temperature with lag demand

# These columns only have a few categories
#
# => onehot encoding is okay
columns_to_onehot_encode = [
    'pickup_dow',
    'time_of_day',
]

# These columns have LOADS of categories
#
# => ordinal encoding is needed             <---- not doing this gives 500GB encoded dataset
#``
# (tried TargetEncoder, but still ended up at 22GB - sticking to ordinal)
columns_to_ordinal_encode = [
    'pickup_zone',
    'route',
    'holiday',
]



# Split `y` BEFORE pipeline
y = df['total_ride_count']
# placeholder = df['avg_fare_amount']
X = df.drop(columns=['total_ride_count'])

# Pipeline
#
# Transfroms the categorical columns to onehot-encoded types
pipeline = ColumnTransformer(
    transformers=[
        # `sparse_output=True` stops the encoder duplicating data
        ("onehot", OneHotEncoder(sparse_output=True), columns_to_onehot_encode), 
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_ordinal_encode),
    ],
    remainder='passthrough'
)
# pipeline.set_output(transform='pandas') # returning pandas cost too much memory
X_encoded = pipeline.fit_transform(X)

# Train vs Test
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=seed)

# Train vs Validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=seed)

***
# SECTION 1: EXPLORATORY DATA ANALYSIS
***

# **TO DO**
1. explain weather trends (below are all important variables)
    - highest temperate
    - cloud coverage
    - precipiation
    - max wind speed
2. explain time of day
    - Bhaskar's heat map
3. expain large monthly trends
4. explain routes
    - most important routes: Upper East Side (South -> North & North -> South are biggest by far)

## **[1.1]** Weather Trends

## **[1.2]** Time of Day

## **[1.3]** Monthly Trends

## **[1.4]** Pickup zones, drop-off zones, and routes

***
# SECTION 2: RIDE DEMAND
***

A total of 3 regressor models were trained, in order to predict `ride demand`:
| Model | Reason | Package |
| ----- | ------ | ------- |
| Random Forest | used as a foundation, given the easy-to-understand nature of decision trees | sklearn |
| Light Gradient Boosted Machine | heavily optimized for largescale data, and more suitable for stronger predictions | lightgbm |
| MLP Neural Net | comparison to deep learning approach | sklearn |

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Analysis helper function
#
# 1. Reports MAE, MSE, and R squared scores
# 2. Plots Residuals vs. Predictions, and Actual vs. Predictions
def analysis(y_true, y_pred):
    # Mean Absolute Error
    mae = mean_absolute_error(y_true, y_pred)
    print(f"Mean Absolute Error: \t{mae:.4f}")

    # Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)
    print(f"Mean Squared Error: \t{mse:.4f}")

    # R squared
    r2 = r2_score(y_true, y_pred)
    print(f"R squared: \t\t{r2:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Residuals
    residuals = y_true - y_pred
    ax[0].scatter(y_pred, residuals, alpha=.4)
    ax[0].axhline(0, color='black', linestyle='--')
    ax[0].set_xlabel("Predicted")
    ax[0].set_ylabel("Residuals")
    ax[0].set_title("Residuals")

    # Actual vs predicted
    ax[1].scatter(y_true, y_pred, alpha=.4)
    ax[1].set_xlabel("Actual")
    ax[1].set_ylabel("Predicted")
    ax[1].set_title("Actual vs. Predicted")

## **[2.1]** Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Fine-tuned model
rf = RandomForestRegressor(
    n_jobs=1,           # parallel compute tends to crash memory (keep it at 1)
    n_estimators=75,    # need reduced processing
    min_samples_leaf=2, # weird clustering around `250` & `400` predictions without this
    max_depth=25,       # maxdepth=25 is large, but there are a lot of features, and validation/testing remain similar to training
    #
    # criterion='absolute_error'        # very low values being mis-assigned    <-------    took too long to compute
)
rf.fit(X_train, y_train)

# Training results
training_predictions = rf.predict(X_train)
analysis(y_train, training_predictions)

# Validation Results
validation_predictions = rf.predict(X_val)
analysis(y_val, validation_predictions)

# Test Results
test_predictions = rf.predict(X_test)
analysis(y_test, test_predictions)

# Clean up (for memory)
del rf
del training_predictions
del validation_predictions
del test_predictions

## **[2.2]** Light GBM

In [ ]:
# from lightgbm import LGBMRegressor

# lg = LGBMRegressor(
#     random_state=seed,
#     learning_rate=0.01,
#     n_estimators=1000,
#     num_leaves=60, # default=31 <--- using alot more leaves than usual 
#     max_depth=-1, # unlimited tree depth (this algorithm is focused on width, more than depth)

#     # The type of feature importance to be filled into feature_importances_. 
#     # If ‘split’, result contains numbers of times the feature is used in a model. 
#     #
#     # If ‘gain’, result contains total gains of splits which use the feature.
#     # importance_type='split',
#     importance_type='gain',
# )
# lg.fit(X_train, y_train)

# # Training results
# training_predictions = lg.predict(X_train)
# analysis(y_train, training_predictions)

# # Validation Results
# validation_predictions = lg.predict(X_val)
# analysis(y_val, validation_predictions)

# # Test Results
# test_predictions = lg.predict(X_test)
# analysis(y_test, test_predictions)

# # Clean up (for memory)
# del lg
# del training_predictions
# del validation_predictions
# del test_predictions

## **[2.3]** MLP Neural Net

In [ ]:
from sklearn.neural_network import MLPRegressor

nn = MLPRegressor(
    random_state=seed, 
    activation='relu',
    max_iter=2000, # the algo needs a lot of time to learn
    tol=0.1 # tolerance
)
nn.fit(X_train, y_train)

# Training results
training_predictions = nn.predict(X_train)
analysis(y_train, training_predictions)

# Validation Results
validation_predictions = nn.predict(X_val)
analysis(y_val, validation_predictions)

# Test Results
test_predictions = nn.predict(X_test)
analysis(y_test, test_predictions)

# Clean up (for memory)
del nn
del training_predictions
del validation_predictions
del test_predictions

***
# SECTION 3: FARE AMOUNT
***

A total of 6 regressor models were trained:

## 3 models for predicting the **fare amount of each ride**:
1. Linear
2. Random Forest
3. MLP Neural Net

## 3 models for predicting the **total revenue**:
4. Linear
5. Random Forest
6. MLP Neural Net

## **[3.1]** Fare Amount: Linear

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler

# Ride-level fare dataset derived from the shared base table
fare_df = read_fare_data(month_start=1, month_end=12)
fare_df_sample = fare_df.sample(n=500000, random_state=seed)
fare_df = fare_df.astype({
    'pickup_zone': 'category',
    'dropoff_zone': 'category',
    'route': 'category',
    'time_of_day': 'category',
    'holiday': 'category',
    'payment_type': 'category',
})

fare_df = fare_df.drop(errors='ignore', columns=[
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'pickup_date',
    'pickup_day',
    'pickup_week',
    'pickup_year',
    'total_amount',
    'extra',
    'mta_tax',
    'tip_amount',
    'tolls_amount',
    'improvement_surcharge',
    'Airport_fee',
    'congestion_surcharge',
])

fare_columns_to_onehot_encode = [
    'pickup_dow',
    'pickup_month',
    'time_of_day',
    'payment_type',
]

fare_columns_to_ordinal_encode = [
    'pickup_zone',
    'dropoff_zone',
    'route',
    'holiday',
]

fare_y = fare_df['fare_amount']
fare_X = fare_df.drop(columns=['fare_amount'])

fare_pipeline = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(sparse_output=True, handle_unknown='ignore'), fare_columns_to_onehot_encode),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), fare_columns_to_ordinal_encode),
    ],
    remainder='passthrough'
)

fare_X_encoded = fare_pipeline.fit_transform(fare_X)

fare_X_train, fare_X_test, fare_y_train, fare_y_test = train_test_split(
    fare_X_encoded, fare_y, test_size=0.2, random_state=seed
)
fare_X_train, fare_X_val, fare_y_train, fare_y_val = train_test_split(
    fare_X_train, fare_y_train, test_size=0.3, random_state=seed
)

fare_linear = Pipeline(steps=[
    ("scale", MaxAbsScaler()),
    ("model", LinearRegression()),
])
fare_linear.fit(fare_X_train, fare_y_train)

training_predictions = fare_linear.predict(fare_X_train)
analysis(fare_y_train, training_predictions)

validation_predictions = fare_linear.predict(fare_X_val)
analysis(fare_y_val, validation_predictions)

test_predictions = fare_linear.predict(fare_X_test)
analysis(fare_y_test, test_predictions)

# Clean up (for memory)
del fare_linear
del training_predictions
del validation_predictions
del test_predictions

## **[3.2]** Fare Amount: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Fine-tuned model
fare_rf = RandomForestRegressor(
    random_state=seed,
    n_jobs=1,            # parallel compute tends to crash memory (keep it at 1)
    n_estimators=120,    # enough trees to stabilise fares without dragging runtime too far
    min_samples_leaf=3,  # smooths very low / very high individual fare predictions
    max_depth=28,        # fares are still nonlinear, so allow deeper trees
    max_features='sqrt',
)
fare_rf.fit(fare_X_train, fare_y_train)

# Training results
training_predictions = fare_rf.predict(fare_X_train)
analysis(fare_y_train, training_predictions)

# Validation Results
validation_predictions = fare_rf.predict(fare_X_val)
analysis(fare_y_val, validation_predictions)

# Test Results
test_predictions = fare_rf.predict(fare_X_test)
analysis(fare_y_test, test_predictions)

# Clean up (for memory)
del fare_rf
del training_predictions
del validation_predictions
del test_predictions


## **[3.3]** Fare Amount: MLP Neural Net

In [ ]:
from sklearn.neural_network import MLPRegressor

# Neural nets benefit from scaling much more than trees do
fare_nn = Pipeline(steps=[
    ("scale", MaxAbsScaler()),
    ("model", MLPRegressor(
        random_state=seed,
        hidden_layer_sizes=(128, 64),
        activation='relu',
        alpha=1e-4,
        learning_rate_init=0.001,
        max_iter=600,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
    )),
])
fare_nn.fit(fare_X_train, fare_y_train)

# Training results
training_predictions = fare_nn.predict(fare_X_train)
analysis(fare_y_train, training_predictions)

# Validation Results
validation_predictions = fare_nn.predict(fare_X_val)
analysis(fare_y_val, validation_predictions)

# Test Results
test_predictions = fare_nn.predict(fare_X_test)
analysis(fare_y_test, test_predictions)

# Clean up (for memory)
del fare_nn
del fare_df
del training_predictions
del validation_predictions
del test_predictions


***
# SECTION 4: REVENUE PREDICTIONS
***

Predicting for the **first quarter of 2025**, we can:
- take the average of previous weather results for January (2022, 2023, 2024)
- use the last revenue day of 2024
- update the calendar for 2025

## **[4.1]** Revenue Dataset

In [ ]:
##############################################################################
# Revenue dataset derived from the same shared base table used above
##############################################################################

revenue_df = read_revenue_data(month_start=1, month_end=12)

revenue_df = revenue_df.astype({
    'pickup_zone': 'category',
    'dropoff_zone': 'category',
    'route': 'category',
    'time_of_day': 'category',
    'holiday': 'category',
})

# For "expected revenue", keep the model honest:
# - do NOT feed actual total_revenue back in
# - do NOT use realised avg_fare_amount as a feature, because in practice that
#   would need to come from the fare model above
revenue_df = revenue_df.drop(errors='ignore', columns=[
    'pickup_date',
    'pickup_day',
    'pickup_week',
])

revenue_columns_to_onehot_encode = [
    'pickup_dow',
    'pickup_month',
    'time_of_day',
]

revenue_columns_to_ordinal_encode = [
    'pickup_zone',
    'dropoff_zone',
    'route',
    'holiday',
]

revenue_y = revenue_df['total_revenue']
revenue_X = revenue_df.drop(columns=['total_revenue', 'avg_fare_amount'])

revenue_pipeline = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(sparse_output=True, handle_unknown='ignore'), revenue_columns_to_onehot_encode),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), revenue_columns_to_ordinal_encode),
    ],
    remainder='passthrough'
)

revenue_X_encoded = revenue_pipeline.fit_transform(revenue_X)

revenue_X_train, revenue_X_test, revenue_y_train, revenue_y_test = train_test_split(
    revenue_X_encoded, revenue_y, test_size=0.2, random_state=seed
)
revenue_X_train, revenue_X_val, revenue_y_train, revenue_y_val = train_test_split(
    revenue_X_train, revenue_y_train, test_size=0.3, random_state=seed
)







## **[4.2]** Revenue: Linear

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler

revenue_linear = Pipeline(steps=[
    ("scale", MaxAbsScaler()),
    ("model", LinearRegression()),
])
revenue_linear.fit(revenue_X_train, revenue_y_train)

# Training results
training_predictions = revenue_linear.predict(revenue_X_train)
analysis(revenue_y_train, training_predictions)

# Validation Results
validation_predictions = revenue_linear.predict(revenue_X_val)
analysis(revenue_y_val, validation_predictions)

# Test Results
test_predictions = revenue_linear.predict(revenue_X_test)
analysis(revenue_y_test, test_predictions)
# Clean up (for memory)
del revenue_linear
del training_predictions
del validation_predictions
del test_predictions


## **[4.3]** Revenue: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

revenue_rf = RandomForestRegressor(
    random_state=seed,
    n_jobs=1,
    n_estimators=150,
    min_samples_leaf=2,
    max_depth=30,
    max_features='sqrt',
)
revenue_rf.fit(revenue_X_train, revenue_y_train)

# Training results
training_predictions = revenue_rf.predict(revenue_X_train)
analysis(revenue_y_train, training_predictions)

# Validation Results
validation_predictions = revenue_rf.predict(revenue_X_val)
analysis(revenue_y_val, validation_predictions)

# Test Results
test_predictions = revenue_rf.predict(revenue_X_test)
analysis(revenue_y_test, test_predictions)
# Clean up (for memory)
del revenue_rf
del training_predictions
del validation_predictions
del test_predictions

## **[4.4]** Revenue: MLP Neural Net

In [ ]:
from sklearn.neural_network import MLPRegressor

revenue_nn = Pipeline(steps=[
    ("scale", MaxAbsScaler()),
    ("model", MLPRegressor(
        random_state=seed,
        hidden_layer_sizes=(128, 64),
        activation='relu',
        alpha=1e-4,
        learning_rate_init=0.001,
        max_iter=700,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
    )),
])
revenue_nn.fit(revenue_X_train, revenue_y_train)

# Training results
training_predictions = revenue_nn.predict(revenue_X_train)
analysis(revenue_y_train, training_predictions)

# Validation Results
validation_predictions = revenue_nn.predict(revenue_X_val)
analysis(revenue_y_val, validation_predictions)

# Test Results
test_predictions = revenue_nn.predict(revenue_X_test)
analysis(revenue_y_test, test_predictions)

# Clean up (for memory)
del revenue_nn
del revenue_df
del training_predictions
del validation_predictions
del test_predictions
del 

## **[4.5]** Expected Fare and Revenue for Q1 2025

In [ ]:
##############################################################################
#
# Forecast template for Q1 2025
#
# The "expected revenue" estimate is built from:
#     expected revenue = expected demand * expected fare
#
# Using the strongest model from each section is usually the best option.
# Swap `rf` / `lg` / `nn` or `fare_linear` / `fare_rf` / `fare_nn` if your
# validation scores show a different winner.
#
def build_q1_2025_calendar_template():
    future_dates = pd.date_range("2025-01-01", "2025-03-31", freq="D")
    q1 = pd.DataFrame({'pickup_date': future_dates})
    q1['pickup_month'] = q1['pickup_date'].dt.month
    q1['pickup_week'] = q1['pickup_date'].dt.isocalendar().week.astype(int)
    q1['pickup_dow'] = q1['pickup_date'].dt.weekday

    cal = read_calendar_data()
    q1 = pd.merge(q1, cal, left_on='pickup_date', right_on='date', how='left')
    q1['holiday'] = q1['holiday'].fillna("None")
    q1 = q1.drop(columns=['date', 'holiday_type'])

    return q1

def average_weather_for_q1_2025():
    weather = read_weather_data().copy()
    weather['month_day'] = weather['date'].dt.strftime('%m-%d')

    q1_days = pd.date_range("2025-01-01", "2025-03-31", freq="D")
    q1_weather = pd.DataFrame({'pickup_date': q1_days})
    q1_weather['month_day'] = q1_weather['pickup_date'].dt.strftime('%m-%d')

    numeric_cols = weather.select_dtypes(include=[np.number, bool]).columns.tolist()
    grouped_weather = weather.groupby('month_day', as_index=False)[numeric_cols].mean()

    q1_weather = pd.merge(q1_weather, grouped_weather, on='month_day', how='left')
    q1_weather = q1_weather.drop(columns=['month_day'])

    return q1_weather

# Start from the routes / time-of-day combinations seen in 2024
route_template = revenue_df[['pickup_zone', 'dropoff_zone', 'route', 'time_of_day']].drop_duplicates()
calendar_template = build_q1_2025_calendar_template()
weather_template = average_weather_for_q1_2025()

q1_2025 = calendar_template.merge(weather_template, on='pickup_date', how='left')
q1_2025 = q1_2025.merge(route_template, how='cross')

# Use the last observed demand for each route + time-of-day as the lag signal
last_lag = (
    revenue_df.sort_values('pickup_month')
    [['route', 'time_of_day', 'lag_demand']]
    .dropna()
    .groupby(['route', 'time_of_day'], as_index=False)
    .tail(1)
)
q1_2025 = q1_2025.merge(last_lag, on=['route', 'time_of_day'], how='left')
q1_2025['lag_demand'] = q1_2025['lag_demand'].fillna(0)

# Simple placeholders for route-level trip shape
route_trip_stats = (
    revenue_df.groupby(['pickup_zone', 'dropoff_zone', 'route'], as_index=False)
    .agg(
        avg_trip_distance=('avg_trip_distance', 'mean'),
        avg_duration_mins=('avg_duration_mins', 'mean'),
    )
)
q1_2025 = q1_2025.merge(route_trip_stats, on=['pickup_zone', 'dropoff_zone', 'route'], how='left')


# Match the weather transformations used in the demand model
q1_2025['precip'] = q1_2025['lag_demand'] * q1_2025['precip']
q1_2025['temp_high'] = q1_2025['lag_demand'] * q1_2025['temp_high']

# ----- Expected demand -----
q1_demand_features = q1_2025[df.drop(columns=['total_ride_count']).columns]
q1_demand_encoded = pipeline.transform(q1_demand_features)
#q1_2025['expected_ride_count'] = np.maximum(lg.predict(q1_demand_encoded), 0)

# ----- Expected fare -----
q1_fare_features = q1_2025[fare_X.columns]
q1_fare_encoded = fare_pipeline.transform(q1_fare_features)
q1_2025['expected_fare_amount'] = np.maximum(fare_rf.predict(q1_fare_encoded), 0)

# ----- Expected revenue -----
q1_2025['expected_revenue'] = q1_2025['expected_ride_count'] * q1_2025['expected_fare_amount']

q1_2025.groupby('pickup_month', as_index=False).agg(
    expected_ride_count=('expected_ride_count', 'sum'),
    expected_revenue=('expected_revenue', 'sum'),
).round(2)